# Explore Silver v=3 Data\n\nRead compacted Parquet files from S3 and inspect the schema.

In [1]:
import boto3
import io
import pyarrow.parquet as pq
import pandas as pd

s3 = boto3.client("s3")
BUCKET = "prediction-markets-data"

def read_silver_v3(event_type: str, date: str) -> pd.DataFrame:
    """Read compacted silver v=3 Parquet for a given event type and date."""
    prefix = f"silver/kalshi_ws/{event_type}/date={date}/v=3/"
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".parquet")]
    if not keys:
        print(f"No files found at {prefix}")
        return pd.DataFrame()
    
    tables = []
    for k in keys:
        body = s3.get_object(Bucket=BUCKET, Key=k)["Body"].read()
        tables.append(pq.read_table(io.BytesIO(body)))
    
    import pyarrow as pa
    return pa.concat_tables(tables).to_pandas()

## OrderBookUpdate

In [2]:
ob = read_silver_v3("OrderBookUpdate", "2026-04-26")
print(f"Shape: {ob.shape}")
print(f"\nDtypes:\n{ob.dtypes}")
print(f"\nHead:")
ob.head(10)

Shape: (5836226, 6)

Dtypes:
t_receipt_ns        int64
market_ticker    category
bid_yes             int32
ask_yes             int32
bid_size            int32
ask_size            int32
dtype: object

Head:


,t_receipt_ns,market_ticker,bid_yes,ask_yes,bid_size,ask_size
0,1777161560496745216,KXNBAPTS-26APR25NYKATL-ATLJJOHNSON1-15,31,68,65,150
1,1777161560497138176,KXNBAGAME-26APR25NYKATL-NYK,96,97,55648,842129
2,1777161560497299200,KXNBAPTS-26APR25NYKATL-NYKJBRUNSON11-20,44,64,63,77
3,1777161560540026368,KXNBAPTS-26APR25NYKATL-NYKJBRUNSON11-20,44,64,63,77
4,1777161560577339392,KXNBAPTS-26APR25NYKATL-NYKJBRUNSON11-20,44,64,63,77
5,1777161560659948032,KXNBAPTS-26APR25NYKATL-NYKJBRUNSON11-20,44,64,63,77
6,1777161560675779328,KXNBAPTS-26APR25NYKATL-NYKJBRUNSON11-20,44,64,63,77
7,1777161560675950848,KXNBAPTS-26APR25NYKATL-NYKJBRUNSON11-20,44,64,63,77
8,1777161560677326080,KXNBAPTS-26APR25NYKATL-NYKJHART3-15,16,99,300,301
9,1777161560677424640,KXNBAPTS-26APR25NYKATL-NYKJHART3-15,16,99,300,301


## TradeEvent

In [3]:
trades = read_silver_v3("TradeEvent", "2026-04-26")
print(f"Shape: {trades.shape}")
print(f"\nDtypes:\n{trades.dtypes}")
print(f"\nHead:")
trades.head(10)

Shape: (596595, 5)

Dtypes:
t_receipt_ns        int64
market_ticker    category
side             category
price               int32
size                int32
dtype: object

Head:


,t_receipt_ns,market_ticker,side,price,size
0,1777161560633037824,KXNBASPREAD-26APR25NYKATL-NYK11,yes,70,147
1,1777161560711417344,KXNBAGAME-26APR25DENMIN-DEN,yes,52,92
2,1777161560772158208,KXNBASPREAD-26APR25NYKATL-NYK5,no,89,185
3,1777161560788641792,KXNBAGAME-26APR25NYKATL-NYK,no,96,147
4,1777161560863396608,KXNBAGAME-26APR25NYKATL-ATL,yes,4,84
5,1777161560863440640,KXNBAGAME-26APR25NYKATL-ATL,yes,4,853
6,1777161560985740544,KXNBAGAME-26APR25DENMIN-DEN,yes,52,50
7,1777161561008275200,KXNBAGAME-26APR25NYKATL-ATL,yes,4,33
8,1777161561027691264,KXNBASPREAD-26APR25NYKATL-NYK14,yes,57,23
9,1777161561027785984,KXNBASPREAD-26APR25NYKATL-NYK14,yes,57,150


## MMFillEvent

In [4]:
fills = read_silver_v3("MMFillEvent", "2026-04-26")
print(f"Shape: {fills.shape}")
print(f"\nDtypes:\n{fills.dtypes}")
print(f"\nHead:")
fills.head(10)

Shape: (448, 11)

Dtypes:
t_receipt_ns               int64
market_ticker           category
side                    category
price                      int32
fill_size                  int32
order_remaining_size       int32
position_before            int32
position_after             int32
maker_fee                  int32
order_id                     str
book_mid_at_fill           int32
dtype: object

Head:


,t_receipt_ns,market_ticker,side,price,fill_size,order_remaining_size,position_before,position_after,maker_fee,order_id,book_mid_at_fill
0,1777161565171942144,KXNBAPTS-26APR25NYKATL-NYKKTOWNS32-15,buy,69,1,0,0,1,1,paper-52283,0
1,1777161571090477568,KXNBAPTS-26APR25NYKATL-ATLOOKONGWU17-10,sell,73,1,0,0,-1,1,paper-52311,0
2,1777161598991067904,KXNBAPTS-26APR25NYKATL-NYKKTOWNS32-15,buy,63,1,0,1,2,1,paper-52403,0
3,1777161621155176960,KXNBAPTS-26APR25NYKATL-NYKJHART3-15,sell,35,1,0,0,-1,1,paper-52487,0
4,1777161823313049088,KXNBAPTS-26APR25NYKATL-ATLCMCCOLLUM3-25,buy,23,1,0,-1,0,1,paper-52967,0
5,1777161829738935296,KXNBAPTS-26APR25NYKATL-ATLJJOHNSON1-15,sell,73,1,0,0,-1,1,paper-52982,0
6,1777161996210719232,KXNBAPTS-26APR25NYKATL-NYKKTOWNS32-15,sell,99,1,0,2,1,1,paper-53357,0
7,1777162129981808384,KXNBAPTS-26APR25NYKATL-NYKJBRUNSON11-25,buy,1,1,0,5,6,1,paper-51093,0
8,1777162182941332992,KXNBAPTS-26APR25NYKATL-NYKJBRUNSON11-25,buy,1,1,0,6,7,1,paper-53909,0
9,1777162197117702144,KXNBAPTS-26APR25NYKATL-NYKKTOWNS32-20,sell,77,1,0,1,0,1,paper-54127,0


## Parquet Schema (raw Arrow, before pandas conversion)

In [5]:
# Read the raw Parquet schema to verify dictionary encoding and types
prefix = "silver/kalshi_ws/OrderBookUpdate/date=2026-04-26/v=3/"
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
key = [o["Key"] for o in resp["Contents"] if o["Key"].endswith(".parquet")][0]
body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
pf = pq.ParquetFile(io.BytesIO(body))

print("Arrow schema:")
print(pf.schema_arrow)
print(f"\nNum row groups: {pf.metadata.num_row_groups}")
print(f"Num rows: {pf.metadata.num_rows}")
print(f"File size: {len(body):,} bytes")

Arrow schema:
t_receipt_ns: int64
market_ticker: dictionary<values=string, indices=int16, ordered=0>
bid_yes: int32
ask_yes: int32
bid_size: int32
ask_size: int32

Num row groups: 59
Num rows: 5836226
File size: 56,358,180 bytes
